In [ ]:
!pip install pandas numpy plotly requests yfinance lxml mstarpy==11.1.0 -q

import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from datetime import date
import os, warnings
warnings.filterwarnings("ignore")

def fmt_pct(v):
    if v is None or (isinstance(v, float) and pd.isna(v)): return "N/D"
    return f"{'🟢' if v >= 0 else '🔴'} {v:+.2f}%"
def fmt_eur(v):
    if v is None or (isinstance(v, float) and pd.isna(v)): return "N/D"
    return f"€ {v:,.2f}"
def fmt_gp(v):
    if v is None or (isinstance(v, float) and pd.isna(v)): return "N/D"
    return f"{'🟢' if v >= 0 else '🔴'} € {v:+,.2f}"


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


### Change Working Directory

To allow the `%run` commands to find the required `.ipynb` files, we need to change the current working directory to the subfolder where these files are located. If the folder name contains spaces, it must be enclosed in quotation marks.

In [ ]:
import os

# Change the current working directory to the subfolder
# If this is not the correct path, please update it.
subfolder_path = "/content/drive/My Drive/Colab Notebooks/Finanza"

# Check if the directory exists before changing
if os.path.isdir(subfolder_path):
    %cd "/content/drive/My Drive/Colab Notebooks/Finanza"
    print(f"Changed directory to: {os.getcwd()}")
else:
    print(f"Error: The subfolder '{subfolder_path}' was not found. Please ensure it exists and the path is correct.")
    print("Current working directory: ", os.getcwd())

In [ ]:
# Quotazioni condivise con GitHub e Streamlit; za usa il metodo Untitled64.
import tempfile, sys, importlib, requests, json
from pathlib import Path
from datetime import date
DATA_INIZIO = date(date.today().year - 1, 12, 31)
DATA_FINE = date.today()
repo_url = "https://raw.githubusercontent.com/leosorge/Cruscotto/main/"
code_dir = Path(tempfile.mkdtemp(prefix="cruscotto-"))
for name in ["quotes.py", "fund_sources.py", "morningstar_process.py", "morningstar_worker.py", "morningstar_search.py", "titoli.json"]:
    response = requests.get(repo_url + name, timeout=30)
    response.raise_for_status()
    (code_dir / name).write_text(response.text, encoding="utf-8")
sys.path.insert(0, str(code_dir))
for module in ["quotes", "fund_sources", "morningstar_process"]:
    sys.modules.pop(module, None)
from quotes import fetch_asset
assets = json.loads((code_dir / "titoli.json").read_text())
rows = []
for asset in assets:
    print(asset['bank'], asset['name'])
    try:
        rows.append(fetch_asset(asset, DATA_INIZIO, DATA_FINE))
    except Exception as exc:
        rows.append({'Titolo': asset['name'], 'Banca': asset['bank'], 'ISIN': asset.get('isin'), 'Stato': 'Errore', 'Dettaglio': str(exc)})
df_tutti = pd.DataFrame(rows).rename(columns={
    'Prezzo iniziale': 'Prezzo_1gen', 'Ultimo prezzo': 'Prezzo_oggi',
    'Variazione %': 'Perf_YTD%'})
for column in ['Prezzo_1gen', 'Prezzo_oggi', 'Perf_YTD%']:
    if column not in df_tutti:
        df_tutti[column] = float('nan')
display(df_tutti)
df_tutti.to_csv("report_cruscotto.csv", index=False)
print("Quantità non caricate: confronto di prezzi, non valore del patrimonio.")


In [ ]:
# ─── DASHBOARD PERFORMANCE % ────────────────────────────────

if "df_tutti" not in dir() or df_tutti.empty:
    print("⚠️  Carica prima i dati")
else:
    today_str = date.today().strftime("%d/%m/%Y")
    df_ok = df_tutti[df_tutti["Perf_YTD%"].notna()].copy()
    df_ok = df_ok.sort_values(["Banca","Perf_YTD%"], ascending=[True,True])

    fig = go.Figure(go.Bar(
        x=df_ok["Perf_YTD%"], y=df_ok["Titolo"], orientation="h",
        marker_color=[("#27ae60" if v >= 0 else "#e74c3c") for v in df_ok["Perf_YTD%"]],
        text=[f"{v:+.2f}%" for v in df_ok["Perf_YTD%"]], textposition="outside",
        customdata=df_ok[["Banca","Prezzo_1gen","Prezzo_oggi"]].values,
        hovertemplate="<b>%{y}</b><br>Banca: %{customdata[0]}<br>1/gen: %{customdata[1]:.4f}<br>Oggi: %{customdata[2]:.4f}<br><b>%{x:+.2f}%</b><extra></extra>",
    ))
    fig.add_vline(x=0, line_width=1.5, line_color="black")
    fig.update_layout(
        title=f"📊 Performance YTD — {today_str}",
        xaxis_title="Variazione % da inizio anno",
        height=max(500, len(df_ok)*32), margin=dict(l=280,r=130),
        paper_bgcolor="white", plot_bgcolor="#f8f9fa", showlegend=False,
    )
    fig.show()

    if "Perf_custom%" in df_tutti.columns:
        df_cu = df_tutti[df_tutti["Perf_custom%"].notna()].sort_values("Perf_custom%", ascending=True)
        if not df_cu.empty:
            fig2 = go.Figure(go.Bar(
                x=df_cu["Perf_custom%"], y=df_cu["Titolo"], orientation="h",
                marker_color=[("#27ae60" if v >= 0 else "#e74c3c") for v in df_cu["Perf_custom%"]],
                text=[f"{v:+.2f}%" for v in df_cu["Perf_custom%"]], textposition="outside",
            ))
            fig2.add_vline(x=0, line_width=1.5, line_color="black")
            fig2.update_layout(title="📊 Performance dalla data personalizzata",
                               height=max(500,len(df_cu)*32), margin=dict(l=280,r=130))
            fig2.show()

    print("\n📈 Riepilogo per banca:")
    print(df_tutti.groupby("Banca")["Perf_YTD%"]
          .agg(Media="mean", Min="min", Max="max").round(2).to_string())


In [ ]:
# ─── VALORIZZAZIONE TOTALE IN EURO ──────────────────────────

if "df_tutti" not in dir() or df_tutti.empty:
    print("⚠️  Carica prima i dati")
elif "Quantità" not in df_tutti.columns:
    print("Lista di osservazione: quantità non caricate, nessuna valorizzazione patrimoniale.")
else:
    today_str = date.today().strftime("%d/%m/%Y")
    df_v = df_tutti[(df_tutti["Quantità"].notna()) & (df_tutti["Quantità"] > 0)].copy()

    if df_v.empty:
        print("⚠️  Tutte le quantità sono 0 — aggiorna i file Quantita_*.ipynb")
    else:
        grand_oggi = df_v["Valore_oggi €"].sum(skipna=True)
        grand_1gen = df_v["Valore_1gen €"].sum(skipna=True)
        grand_gp   = df_v["G/P €"].sum(skipna=True)
        pct_ytd    = (grand_oggi - grand_1gen) / grand_1gen * 100 if grand_1gen else 0

        print(f"\n💰 PORTAFOGLIO TOTALE al {today_str}")
        print("=" * 65)
        for banca, grp in df_v.groupby("Banca"):
            t = grp["Valore_oggi €"].sum(skipna=True)
            g = grp["G/P €"].sum(skipna=True)
            print(f"  {banca:15s}: € {t:>12,.2f}   G/P: {fmt_gp(g)}")
        print("-" * 65)
        print(f"  {'TOTALE':15s}: € {grand_oggi:>12,.2f}   G/P: {fmt_gp(grand_gp)}")
        pct_str = f"{'🟢' if pct_ytd >= 0 else '🔴'} {pct_ytd:+.2f}%"
        print(f"  {'Perf. totale':15s}: {pct_str}")
        print("=" * 65)

        df_disp = df_v[["Titolo","Banca","Quantità","Prezzo_1gen","Prezzo_oggi",
                         "Perf_YTD%","Valore_1gen €","Valore_oggi €","G/P €"]].copy()
        df_disp["Perf_YTD%"]    = df_disp["Perf_YTD%"].apply(fmt_pct)
        df_disp["Valore_1gen €"] = df_disp["Valore_1gen €"].apply(fmt_eur)
        df_disp["Valore_oggi €"] = df_disp["Valore_oggi €"].apply(fmt_eur)
        df_disp["G/P €"]         = df_disp["G/P €"].apply(fmt_gp)
        print()
        display(df_disp)

        # Torta allocazione per banca
        bv = df_v.groupby("Banca")["Valore_oggi €"].sum().reset_index()
        fig_pie = go.Figure(go.Pie(
            labels=bv["Banca"], values=bv["Valore_oggi €"], hole=0.4,
            texttemplate="%{label}<br>€ %{value:,.0f}<br>%{percent}",
            marker_colors=["#1a5276","#1e8449","#922b21"],
        ))
        fig_pie.update_layout(title=f"💼 Allocazione per banca — {today_str}", height=430)
        fig_pie.show()

        # Barre valore per titolo
        df_bar = df_v[df_v["Valore_oggi €"].notna()].sort_values("Valore_oggi €", ascending=True)
        fig_bar = go.Figure(go.Bar(
            x=df_bar["Valore_oggi €"], y=df_bar["Titolo"], orientation="h",
            text=[f"€ {v:,.0f}" for v in df_bar["Valore_oggi €"]], textposition="outside",
            marker_color="#2980b9",
            customdata=df_bar[["Banca","Quantità","Perf_YTD%"]].values,
            hovertemplate="<b>%{y}</b><br>Banca: %{customdata[0]}<br>Qtà: %{customdata[1]}<br>YTD: %{customdata[2]:+.2f}%<br>€ %{x:,.2f}<extra></extra>",
        ))
        fig_bar.update_layout(
            title="📈 Valore per titolo (quantità × prezzo oggi)",
            height=max(450, len(df_bar)*32), margin=dict(l=280,r=130),
            xaxis_title="Valore in €",
        )
        fig_bar.show()
